# Setup: your knowledge graph environment

**Time**: ~15 minutes. **Cost**: Effectively $0 to set up — the graph database has a permanently free tier, and the test calls in this notebook use a handful of tokens on Gemini's cheapest model.

> **Running this locally, or in VS Code instead of Colab?** See [Session 1's setup guide](../week_2/lesson08_setup_guide.ipynb) for how to open any of these notebooks with `uv`, either in a browser tab or inside VS Code. Nothing extra to do if you're in Colab.

This week adds one more moving part to the pipeline you've been building since Session 1. Up to now, every notebook has talked to one thing: an AI model, over an API. This week you're also talking to a **graph database** — a separate system that stores nodes (entities) and the typed relationships between them, and answers questions by traversing those relationships rather than by scanning rows.

That means two accounts instead of one, and two connections to test before you touch any real content. This notebook gets both working: a free graph database instance, and the same kind of swappable LLM connection you set up in Session 1, generalized so the rest of this week's notebook can call whichever model provider you configure with a single line.

## A note on privacy

Same rule as every other session: work only with public example data, synthetic data, or the anonymized examples this course provides. The documents you'll build a graph from this week are entirely invented — a fictional company, fictional customers, fictional tickets — so there's nothing to protect, but the habit matters more than this specific exercise. A knowledge graph built from real internal documents is, by construction, a concentrated store of exactly the kind of information Session 3 covered: names, complaints, internal decisions. Treat a real one accordingly.

Data-use policy for your Gemini key still depends on whether it's billed under a "Paid tier" project, per Session 1 — check the badge in [AI Studio](https://aistudio.google.com) rather than assuming. The graph database provider (Neo4j) is a separate company with its own terms; the free tier used here is a hosted instance on their infrastructure, so don't put anything in it you wouldn't put in any other third-party cloud service.

## Part 1: get a graph database

### What is Neo4j, and why Aura instead of installing something?

**Neo4j** is a graph database — software built specifically to store nodes and relationships and to answer traversal queries efficiently, the way a spreadsheet is built specifically for rows and formulas. **Cypher** is Neo4j's query language, the graph equivalent of SQL; Session 6's overview linked its [getting-started guide](https://neo4j.com/docs/getting-started/cypher/), and you'll write real Cypher later in this notebook and the next one.

Neo4j can run two ways: installed locally (Neo4j Desktop, a program on your own machine), or hosted for you (**Aura**, Neo4j's managed cloud service). This course defaults to Colab specifically to avoid local installation, and Aura is the same default applied to the database: **AuraDB Free** is a permanently free, hosted instance — no credit card, no time limit, created with a Google or GitHub login in about a minute ([Neo4j's own FAQ and pricing pages](https://neo4j.com/docs/aura/)). If you already run Neo4j Desktop locally and would rather point this notebook at that instead, the connection cell below works the same way against a local address (`bolt://localhost:7687`) — the only thing that changes is which URI you put in Secrets. The rest of this guide assumes Aura, since it needs nothing installed and matches the course's Colab-first default.

**What the free tier actually gives you**, and where documentation disagrees: Neo4j's own pages are not fully consistent with each other on the exact ceiling — one source states 50,000 nodes and 175,000 relationships, another states 200,000 nodes and 400,000 relationships. Either number is far more than this week's exercise needs (our graph will have well under 100 nodes), so the discrepancy doesn't matter for coursework, but don't quote a specific number to a stakeholder without checking the current limit in your own Aura console first — this is exactly the "verify before stating as fact" habit the course has been building. One number both sources agree on: a Free instance **auto-pauses after 72 hours of inactivity** (your data is retained, it just needs waking up — open the console and it resumes in under a minute) and is **deleted if left paused for 90 days**. If you come back to this notebook after a break and the connection cell fails, check the Aura console first before assuming something is broken.

### Steps to create your instance

1. Go to [console.neo4j.io](https://console.neo4j.io) and sign in (Google, GitHub, or email — no card required).
2. Click **New instance**, choose **AuraDB Free**.
3. Give it a name (e.g. `ai4tm-session5`) and a region close to you.
4. Click **Create**. Neo4j will show you a **connection URI, username, and a generated password exactly once** — download the credentials file or copy all three now. If you lose the password, you'll need to reset it from the console; Neo4j does not store it for you to look up later.
5. Wait for the instance status to show **Running** (usually under a minute).

## Part 2: store your graph database credentials

Same pattern as Session 1's API key: Colab **Secrets** (the 🔑 icon in the left sidebar), not pasted into a cell. Add three secrets:

- `NEO4J_URI` — starts with `neo4j+s://`, looks like `neo4j+s://xxxxxxxx.databases.neo4j.io`
- `NEO4J_USERNAME` — usually `neo4j`
- `NEO4J_PASSWORD` — the generated password from instance creation

Turn the access toggle on for each, the same as you did for `GEMINI_API_KEY`.

In [ ]:
# Check the three secrets are stored and readable
from google.colab import userdata

for name in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
    try:
        value = userdata.get(name)
        shown = value[:12] + "..." if name == "NEO4J_URI" else "*" * 8
        print(f"{name}: loaded ({shown})")
    except Exception as e:
        print(f"{name}: NOT FOUND -- add it in Secrets. Error: {e}")

## Part 3: install the graph database driver and connect

`neo4j` is Neo4j's official Python driver — the library that turns Python function calls into Cypher queries sent over the network to your Aura instance, the same role `google-genai` plays for Gemini.

In [ ]:
!pip install -q neo4j

print("Driver installed.")

In [ ]:
from neo4j import GraphDatabase
from google.colab import userdata

driver = GraphDatabase.driver(
    userdata.get("NEO4J_URI"),
    auth=(userdata.get("NEO4J_USERNAME"), userdata.get("NEO4J_PASSWORD")),
)

# A minimal Cypher query, just to prove the connection works.
# RETURN behaves like SELECT in SQL: it hands back a value without touching any stored data.
with driver.session() as session:
    result = session.run("RETURN 1 AS ok")
    record = result.single()
    print("Connection ok. Neo4j returned:", record["ok"])

If that errored, check in order: the instance status is **Running** in the Aura console (not paused or still provisioning), the URI starts with `neo4j+s://` with no typos, and the three secret names match exactly (`NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`) with their access toggles on.

## Part 4: a swappable LLM connection

Session 1 set up a single Gemini connection. This week's task adds a requirement: the rest of the pipeline should be able to switch which model provider does the extraction and answering by changing **one config value**, not by rewriting the calling code. That matters for a real reason, not just as an exercise — a company building on an LLM pipeline wants the option to move providers without a rewrite, whether for cost, latency, or a model deprecation.

The pattern: one variable, `LLM_PROVIDER`, and one function, `call_llm(prompt)`, that branches on it. Every other cell in the pipeline notebook calls `call_llm()` and never mentions Gemini, OpenAI, or Anthropic by name directly.

In [ ]:
!pip install -q google-genai

# Uncomment if you're using one of the optional paid providers instead of (or alongside) Gemini:
# !pip install -q openai
# !pip install -q anthropic

print("Gemini library installed.")

In [ ]:
# ---- THE ONE CONFIG VALUE ----
# Change this single line to swap providers everywhere else in this week's notebooks.
LLM_PROVIDER = "gemini"  # one of: "gemini", "openai", "anthropic"
# --------------------------------

from google.colab import userdata


def call_llm(prompt: str, model: str = None) -> str:
    """Send `prompt` to whichever provider LLM_PROVIDER names, return the text response.

    Every notebook this week calls this function instead of a provider's client
    directly, so LLM_PROVIDER is the only thing that needs to change to switch models.
    """
    if LLM_PROVIDER == "gemini":
        from google import genai

        client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
        response = client.models.generate_content(
            model=model or "gemini-3.5-flash-lite",
            contents=prompt,
        )
        return response.text

    elif LLM_PROVIDER == "openai":
        from openai import OpenAI

        client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
        response = client.chat.completions.create(
            model=model or "gpt-4.1-mini",
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

    elif LLM_PROVIDER == "anthropic":
        import anthropic

        client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
        response = client.messages.create(
            model=model or "claude-haiku-4-5-20251001",
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    else:
        raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER!r}")

If you're using `openai` or `anthropic` instead of Gemini, add `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` to Secrets the same way you added `GEMINI_API_KEY` — these are the optional paid alternatives, useful if you already have a key with one of those providers. Model names change over time on every provider; if a call errors mentioning the model name specifically, check that provider's current model list rather than assuming this notebook is wrong.

In [ ]:
# Test the connection through call_llm(), not through a provider client directly --
# this is the function the rest of this week's notebook will use.
test_answer = call_llm("In one sentence, what is a knowledge graph?")
print(test_answer)

## You're set up

Both connections work: a graph database to hold the knowledge graph, and a swappable LLM call for extraction and querying. The next notebook, [`knowledge_graph_pipeline.ipynb`](knowledge_graph_pipeline.ipynb), builds a graph from a set of internal documents and queries it — including asking a model to answer a question using only what the graph actually contains, the technique called GraphRAG.

**Troubleshooting reference:**

| Symptom | Likely cause |
|---|---|
| `NEO4J_URI not found` | Secret name typo, or access toggle off |
| Connection hangs or times out | Instance is paused (open the Aura console to resume it) or still provisioning |
| `Unauthorized` from Neo4j | Password copied incorrectly at creation — reset it from the console |
| `call_llm` errors mentioning a model name | That provider renamed or retired the model; check their current model list |
| Everything worked yesterday, fails today with no code changes | Aura Free auto-pauses after 72 hours idle — check the console status first |